# Gemma 4 Tool-Calling Fine-Tune

**Multi-Platform**: Works on Kaggle, Google Colab, and AMD Dev Cloud

Auto-detects your GPU and adjusts model size + LoRA config:
- **≥24GB VRAM** (A100/MI300X): Gemma 4 31B, r=32
- **15-24GB** (T4/A10G): Gemma 4 31B reduced (r=8, seq=2048)
- **<15GB**: Falls back to Gemma 4 12B

**Setup:**
- Kaggle: Settings → Accelerator → GPU T4x2 or A100
- Colab: Runtime → Change runtime → T4 or A100
- Add `HF_TOKEN` to Kaggle Secrets or Colab Secrets

Dataset: [mtita/gemma4-tool-calling-training](https://huggingface.co/datasets/mtita/gemma4-tool-calling-training)

### CELL 1: Detect Platform & Install

In [ ]:
import os, subprocess# Auto-detect platformIS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not NoneIS_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")IS_AMD = Falsetry:    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()    GPU_NAME = gpu_info.split(",")[0].strip()    GPU_VRAM_MB = int(gpu_info.split(",")[1].strip().replace(" MiB",""))    print(f"🖥️  GPU: {GPU_NAME} ({GPU_VRAM_MB} MiB)")except Exception:    try:        rocm_info = subprocess.check_output(["rocm-smi", "--showmeminfo", "vram"], text=True)        IS_AMD = True        GPU_NAME = "AMD MI300X"        GPU_VRAM_MB = 192 * 1024        print(f"🖥️  GPU: AMD (ROCm detected)")    except:        GPU_NAME = "Unknown"        GPU_VRAM_MB = 16000        print("⚠️  Could not detect GPU, assuming 16GB VRAM")# Decide model size based on available VRAM# 31B 4-bit needs ~20GB; 12B 4-bit needs ~8GBif GPU_VRAM_MB >= 24000:    MODEL_SIZE = "31B"    LORA_R = 32    LORA_ALPHA = 64    BATCH_SIZE = 1    GRAD_ACCUM = 8    MAX_SEQ = 4096    print(f"✅ Enough VRAM for Gemma 4 31B (r={LORA_R})")elif GPU_VRAM_MB >= 15000:    MODEL_SIZE = "31B"    LORA_R = 8    LORA_ALPHA = 16    BATCH_SIZE = 1    GRAD_ACCUM = 16    MAX_SEQ = 2048    print(f"⚠️  Tight VRAM — using Gemma 4 31B with reduced LoRA (r={LORA_R}, seq={MAX_SEQ})")else:    MODEL_SIZE = "12B"    LORA_R = 32    LORA_ALPHA = 64    BATCH_SIZE = 2    GRAD_ACCUM = 4    MAX_SEQ = 4096    print(f"📉 Low VRAM — falling back to Gemma 4 12B")PLATFORM = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else ("AMD" if IS_AMD else "Unknown"))print(f"📍 Platform: {PLATFORM}")print(f"🔧 Config: model={MODEL_SIZE}, r={LORA_R}, batch={BATCH_SIZE}x{GRAD_ACCUM}, seq={MAX_SEQ}")

### CELL 2: Install Dependencies

In [ ]:
# %%captureimport subprocess, sysif IS_AMD:    cmds = [        'pip install "unsloth[amd] @ git+https://github.com/unslothai/unsloth"',        'pip install --no-deps git+https://github.com/huggingface/transformers.git',        'pip install sentencepiece protobuf "datasets>=3.0" "huggingface_hub>=0.34.0"',        'pip install bitsandbytes accelerate peft trl triton',    ]else:    cmds = [        'pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"',        'pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes',        'pip install -q "datasets>=3.0" "huggingface_hub>=0.34.0" sentencepiece protobuf',    ]for cmd in cmds:    print(f"→ {cmd[:60]}...")    subprocess.check_call(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)print("✅ Dependencies installed")

### CELL 3: Load Model (4-bit QLoRA)

In [ ]:
from unsloth import FastModelimport torchMODEL_MAP = {    "31B": "unsloth/gemma-4-31B-it",    "12B": "unsloth/gemma-4-12B-it",}model, tokenizer = FastModel.from_pretrained(    model_name=MODEL_MAP[MODEL_SIZE],    dtype=None,    max_seq_length=MAX_SEQ,    load_in_4bit=True,    full_finetuning=False,)print(f"✅ Loaded {MODEL_MAP[MODEL_SIZE]} in 4-bit")

### CELL 4: Add LoRA Adapters

In [ ]:
model = FastModel.get_peft_model(    model,    finetune_vision_layers=False,    finetune_language_layers=True,    finetune_attention_modules=True,    finetune_mlp_modules=True,    r=LORA_R,    lora_alpha=LORA_ALPHA,    lora_dropout=0,    bias="none",    random_state=3407,)trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)total = sum(p.numel() for p in model.parameters())print(f"✅ LoRA: {trainable:,} trainable / {total:,} total ({trainable/total*100:.2f}%)")

### CELL 5: Load & Prepare Dataset

In [ ]:
from datasets import load_datasetfrom unsloth.chat_templates import get_chat_templatetokenizer = get_chat_template(tokenizer, chat_template="gemma-4")dataset = load_dataset("mtita/gemma4-tool-calling-training",                        data_files="combined_training.jsonl",                        split="train")print(f"📊 Loaded {len(dataset)} training examples")

### CELL 6: Format for Gemma 4 Chat Template

In [ ]:
def formatting_prompts_func(examples):    texts = []    for messages in examples["messages"]:        formatted = []        for msg in messages:            role = msg["role"]            content = msg.get("content", "")            if role == "system":                formatted.append({"role": "user", "content": f"[System Instructions]\n{content}"})                formatted.append({"role": "assistant", "content": "Understood."})            elif role == "user":                formatted.append({"role": "user", "content": content})            elif role == "assistant":                formatted.append({"role": "assistant", "content": content})            elif role == "tool":                tool_name = msg.get("name", "tool")                formatted.append({"role": "user", "content": f"[Tool Result: {tool_name}]\n{content}"})        try:            text = tokenizer.apply_chat_template(                formatted, tokenize=False, add_generation_prompt=False            ).removeprefix('<bos>')            texts.append(text)        except Exception:            texts.append("")    return {"text": texts}dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=2)dataset = dataset.filter(lambda x: len(x["text"]) > 50)print(f"✅ Formatted: {len(dataset)} examples")print(f"📝 Sample:\n{dataset[0]['text'][:300]}")

### CELL 7: Setup Trainer

In [ ]:
from trl import SFTTrainer, SFTConfigfrom unsloth.chat_templates import train_on_responses_onlytrainer = SFTTrainer(    model=model,    tokenizer=tokenizer,    train_dataset=dataset,    eval_dataset=None,    args=SFTConfig(        dataset_text_field="text",        per_device_train_batch_size=BATCH_SIZE,        gradient_accumulation_steps=GRAD_ACCUM,        warmup_steps=20,        num_train_epochs=2,        learning_rate=2e-4,        logging_steps=10,        optim="adamw_8bit",        weight_decay=0.001,        lr_scheduler_type="cosine",        seed=3407,        output_dir="outputs",        report_to="none",        save_strategy="steps",        save_steps=500,        max_seq_length=MAX_SEQ,        fp16=not torch.cuda.is_bf16_supported(),        bf16=torch.cuda.is_bf16_supported(),    ),)trainer = train_on_responses_only(    trainer,    instruction_part="<|turn>user\n",    response_part="<|turn>model\n",)# Verify masking is correctprint("=== Verify: first 200 chars of full text ===")print(tokenizer.decode(trainer.train_dataset[0]["input_ids"])[:200])

### CELL 8: Train!

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)print(f"🚀 Training on {gpu_stats.name} ({max_memory} GB)")print(f"   {len(dataset)} examples × 2 epochs, eff. batch size = {BATCH_SIZE * GRAD_ACCUM}")trainer_stats = trainer.train()used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)train_time = round(trainer_stats.metrics['train_runtime'] / 60, 2)print(f"\n✅ Training complete!")print(f"   Time: {train_time} minutes")print(f"   Peak VRAM: {used_memory} GB ({round(used_memory/max_memory*100, 1)}%)")print(f"   Final loss: {trainer_stats.metrics.get('train_loss', 'N/A')}")

### CELL 9: Test Inference

In [ ]:
FastModel.for_inference(model)test_messages = [    {"role": "user", "content": """You have these tools:- read(path): Read a file- edit(path, old_text, new_text): Replace text in a file- bash(command): Run a shell commandFix the typo in src/utils.py where 'retrun' should be 'return'."""}]inputs = tokenizer.apply_chat_template(    test_messages,    add_generation_prompt=True,    tokenize=True,    return_dict=True,    return_tensors="pt",).to("cuda")from transformers import TextStreamerstreamer = TextStreamer(tokenizer, skip_prompt=True)print("\n🧪 Test inference:")_ = model.generate(    **inputs, streamer=streamer,    max_new_tokens=512, use_cache=False,    temperature=0.7, top_p=0.95, top_k=64,)

### CELL 10: Save & Export

In [ ]:
# HuggingFace token (set yours here or use environment variable)HF_TOKEN = os.environ.get("HF_TOKEN", None)if IS_KAGGLE:    try:        from kaggle_secrets import UserSecretsClient        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")    except: passelif IS_COLAB:    try:        from google.colab import userdata        HF_TOKEN = userdata.get("HF_TOKEN")    except: pass# Save LoRA adapters locallymodel.save_pretrained(f"gemma4-{MODEL_SIZE.lower()}-tool-calling-lora")tokenizer.save_pretrained(f"gemma4-{MODEL_SIZE.lower()}-tool-calling-lora")print("✅ Saved LoRA adapters locally")# Push LoRA to HuggingFaceif HF_TOKEN:    model.push_to_hub(f"mtita/gemma4-{MODEL_SIZE.lower()}-tool-calling-lora", token=HF_TOKEN)    tokenizer.push_to_hub(f"mtita/gemma4-{MODEL_SIZE.lower()}-tool-calling-lora", token=HF_TOKEN)    print("✅ Pushed LoRA to HuggingFace")# Export GGUF for local llama.cppprint("\n⏳ Exporting GGUF (this may take a while)...")model.save_pretrained_gguf(    f"gemma4-{MODEL_SIZE.lower()}-tool-calling-gguf",    tokenizer,    quantization_method="q4_k_m",)print("✅ Exported Q4_K_M GGUF")# Push GGUF to HuggingFaceif HF_TOKEN:    model.push_to_hub_gguf(        f"mtita/gemma4-{MODEL_SIZE.lower()}-tool-calling-Q4_K_M-GGUF",        tokenizer,        quantization_method="q4_k_m",        token=HF_TOKEN,    )    print("✅ Pushed GGUF to HuggingFace")print(f"\n🎉 Done! Download from: https://huggingface.co/mtita/gemma4-{MODEL_SIZE.lower()}-tool-calling-Q4_K_M-GGUF")print("Place the GGUF in ~/fastlane/models/ and update launch-llama-server.sh")